# Replication Notebook — ICU Example, TabPFN (Step 3 of 3)

This notebook reproduces the **TabPFN** row of the ICU results in the paper

*“The choice of reference group can reverse conclusions in the Oaxaca–Blinder decomposition.”*

This is **Step 3** of the replication pipeline and the companion to `icu_139_final_models.ipynb` (Step 2). TabPFN is run **locally** (no hosted API) on the same 139 subsets and base covariates; its outputs complete the **TabPFN row of Table 1 and Appendix Tables 3, 4, 5**.

> See `construct_ICU_data.ipynb` for environment setup, data access, and full pipeline overview. TabPFN is included in `requirements.txt` (`tabpfn==2.0.*`).

## What this notebook produces

- **Phase 1** — TabPFN once per subset (no bootstrap) on all 139 subsets, sign‐flip indicator (`tabpfn_phase1_signflips.csv`).
- **Phase 2** — Bootstrap (**B = 1000**, parallel) on the Phase‐1 flip survivors only (`tabpfn_phase2_bootstrap.csv`).
- **Phase 3** — TabPFN flip‐summary row (`Model | Total flips | 10% | 5% | 1%`) — completes the **TabPFN row of Table 1 and Appendix Tables 4, 5** (`tabpfn_flip_summary_row.csv`).
- **Phase 4** — Bootstrap on HR‐quartile‐2 — completes the **TabPFN row of Appendix Table 3** (HRQ2 coefficient table) (`tabpfn_hrq2_bootstrap.csv`).

## Configuration (Section 1)

- `TABPFN_DEVICE` — `"cpu"`, `"cuda"`, or `"auto"`. On a GPU machine: `"cuda"`. CPU‐only: `"cpu"`.
- `TABPFN_MODEL_PATH` — `"auto"` to download weights to the HuggingFace cache on first fit (requires internet), or a local `.ckpt` path for air‐gapped runs.
- `N_JOBS_BOOT` — parallel bootstrap workers. CPU: `-1` (all cores). Single GPU: `1`. Multi‐GPU: as many workers as GPUs (configure device per worker).

Expected run time: several hours on CPU; ~10 minutes on a single modern GPU.

## How to run

Make sure `ICU_clean.csv` is in the same folder (produced by running Step 1 first), and that `tabpfn` is installed (`pip install -r requirements.txt`). Then execute every cell top to bottom. Outputs are written next to the notebook.


In [2]:
# ============================================================
# SECTION 0 — Imports
# ============================================================
import os, warnings
warnings.filterwarnings("ignore")

# CPU-safety env vars (harmless on GPU too).
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TABPFN_ALLOW_CPU_LARGE_DATASET", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.stats import norm
from joblib import Parallel, delayed

from tabpfn import TabPFNClassifier   # LOCAL package (not tabpfn_client)

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 250)


In [ ]:
# ============================================================
# SECTION 1 — Config  (EDIT THIS SECTION for your machine)
# ============================================================
CONFIG = {
    "random_state": 51,        # must match icu_139_final_models.ipynb for identical subsets
    "min_subset_size": 100,
    "B_boot": 1000,             # bootstrap replicates
    "alpha_levels": [0.10, 0.05, 0.01],
    "n_random_subsamples": 50,
    "random_subsample_fracs": [0.50, 0.30],
}

Y_COL     = "In-hospital_death"
GROUP_COL = "female"            # 0 = men, 1 = women
X_COLS    = ["Age", "ICUType", "HR", "NIMAP", "Temp", "Urine"]  # base covariates

# ------------------------------------------------------------
# TabPFN backend settings 
# ------------------------------------------------------------
# Device:  "cpu", "cuda", or "auto"
TABPFN_DEVICE = "cpu"

# Weights path:
#   "auto" -> TabPFN auto-downloads to HF cache on first fit (requires internet)
#   or an explicit .ckpt path for air-gapped runs.
TABPFN_MODEL_PATH = "auto"
# Example explicit path:
# TABPFN_MODEL_PATH = os.path.expanduser(
#     "~/Library/Caches/tabpfn/tabpfn-v2.6-classifier-v2.6_default.ckpt"
# )

# ------------------------------------------------------------
# Parallelism (EDIT)
# ------------------------------------------------------------
# Bootstrap replicates run in parallel across workers.
# - CPU only:       N_JOBS_BOOT = -1   (all cores; each worker does one fit)
# - Single GPU:     N_JOBS_BOOT = 1    (GPU is the bottleneck)
# - Multi GPU:      N_JOBS_BOOT = <n_gpus>  (+ custom device assignment)
N_JOBS_BOOT = -1 if TABPFN_DEVICE == "cpu" else 1

print("TABPFN_DEVICE    =", TABPFN_DEVICE)
print("TABPFN_MODEL_PATH=", TABPFN_MODEL_PATH)
print("N_JOBS_BOOT      =", N_JOBS_BOOT)


In [ ]:
# ============================================================
# SECTION 2 — Load Data 
# ============================================================
DATA_PATH = "ICU_clean.csv"
df_clean = pd.read_csv(DATA_PATH)
print("Loaded:", df_clean.shape)

needed = X_COLS + [Y_COL, GROUP_COL]
missing = [c for c in needed if c not in df_clean.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}")


In [ ]:
# ============================================================
# SECTION 3 — 139 subsets
# ============================================================
def make_quartile_subsets(df, col, label_prefix):
    q = pd.qcut(df[col], q=4, labels=False, duplicates="drop")
    return [(f"{label_prefix} quartile {int(k)}", df[q == k]) for k in sorted(q.dropna().unique())]

def make_decile_subsets(df, col, label_prefix):
    d = pd.qcut(df[col], q=10, labels=False, duplicates="drop")
    return [(f"{label_prefix} decile {int(k)}", df[d == k]) for k in sorted(d.dropna().unique())]

subsets_raw = []
subsets_raw.append(("All Patients", df_clean))

for t in sorted(df_clean["ICUType"].dropna().unique()):
    subsets_raw.append((f"ICUType {t}", df_clean[df_clean["ICUType"] == t]))

subsets_raw += make_decile_subsets(df_clean, "Age", "Age")
for col, pfx in [("HR", "HR"), ("NIMAP", "MAP"), ("Temp", "Temp"),
                 ("Urine", "Urine"), ("SAPS-I", "SAPS")]:
    if col in df_clean.columns:
        subsets_raw += make_quartile_subsets(df_clean, col, pfx)

subsets_raw.append(("HR > 100",    df_clean[df_clean["HR"] > 100]))
subsets_raw.append(("MAP < 65",    df_clean[df_clean["NIMAP"] < 65]))
subsets_raw.append(("Temp > 38C",  df_clean[df_clean["Temp"] > 38]))
subsets_raw.append(("Urine > 1000", df_clean[df_clean["Urine"] > 1000]))

_rng = np.random.default_rng(CONFIG["random_state"])
for frac in CONFIG["random_subsample_fracs"]:
    for k in range(CONFIG["n_random_subsamples"]):
        seed = int(_rng.integers(1_000_000_000))
        sample = df_clean.sample(frac=frac, replace=False, random_state=seed)
        subsets_raw.append((f"Random {int(frac*100)}% #{k+1}", sample))

subsets = []
for label, sub in subsets_raw:
    sub = sub.dropna(subset=X_COLS + [Y_COL, GROUP_COL]).copy()
    if len(sub) >= CONFIG["min_subset_size"]:
        subsets.append((label, sub))

print(f"Constructed {len(subsets)} subsets (n >= {CONFIG['min_subset_size']}).")
assert len(subsets) == 139, f"Expected 139 subsets, got {len(subsets)}"


In [6]:
# ============================================================
# SECTION 4 — TabPFN helpers: Oaxaca-Blinder + bootstrap
# ============================================================

def build_tabpfn():
    """Fresh TabPFN classifier with the configured weights/device."""
    kwargs = dict(device=TABPFN_DEVICE, ignore_pretraining_limits=True, random_state=None)
    if TABPFN_MODEL_PATH != "auto":
        kwargs["model_path"] = os.path.expanduser(TABPFN_MODEL_PATH)
    return TabPFNClassifier(**kwargs)


def predict_positive_class(model, X):
    p = np.asarray(model.predict_proba(X))
    if p.ndim == 2 and p.shape[1] >= 2:
        return p[:, 1].astype(float)
    return p.ravel().astype(float)


def nonlinear_oaxaca_tabpfn(df, y_col, x_cols, group_col,
                             women_value=1, men_value=0):
    """Twofold nonlinear OBD with separate TabPFN models per group."""
    work = df[x_cols + [y_col, group_col]].dropna()
    women = work[work[group_col] == women_value]
    men   = work[work[group_col] == men_value]
    if women.empty or men.empty:
        raise ValueError("Empty group")

    Xw, yw = women[x_cols], women[y_col].astype(int)
    Xm, ym = men[x_cols],   men[y_col].astype(int)
    if yw.nunique() < 2 or ym.nunique() < 2:
        raise ValueError("One group has only one outcome class")

    mw = build_tabpfn(); mw.fit(Xw, yw)
    mm = build_tabpfn(); mm.fit(Xm, ym)

    y_bar_m = float(ym.mean()); y_bar_w = float(yw.mean())
    M_men_women   = float(predict_positive_class(mw, Xm).mean())   # E_X~men[m_women(X)]
    M_women_men   = float(predict_positive_class(mm, Xw).mean())   # E_X~women[m_men(X)]

    return {
        "gap_men_minus_women":   y_bar_m - y_bar_w,
        "explained_womenref":    M_men_women - y_bar_w,
        "unexplained_womenref":  y_bar_m - M_men_women,
        "explained_menref":      y_bar_m - M_women_men,
        "unexplained_menref":    M_women_men - y_bar_w,
    }


def bootstrap_tabpfn(df, y_col, x_cols, group_col, B, random_state, n_jobs):
    """Pair-bootstrap; workers run in parallel via joblib."""
    rng = np.random.default_rng(random_state)
    base = nonlinear_oaxaca_tabpfn(df, y_col, x_cols, group_col)
    g0 = df[df[group_col] == 0]; g1 = df[df[group_col] != 0]
    n0, n1 = len(g0), len(g1)
    seeds = rng.integers(1_000_000_000, size=B)

    def one(seed):
        r = np.random.default_rng(int(seed))
        bd = pd.concat([
            g0.sample(n0, replace=True, random_state=int(r.integers(1e9))),
            g1.sample(n1, replace=True, random_state=int(r.integers(1e9))),
        ])
        try:
            return nonlinear_oaxaca_tabpfn(bd, y_col, x_cols, group_col)
        except Exception:
            return None

    # return_as="generator" -> per-replicate progress
    gen = Parallel(n_jobs=n_jobs, backend="loky", return_as="generator")(
        delayed(one)(s) for s in seeds
    )
    draws = []
    for b, d in enumerate(gen, start=1):
        if b == 1 or b % 25 == 0 or b == B:
            print(f"    [{b}/{B}]")
        if d is not None:
            draws.append(d)

    def col(k): return np.array([d[k] for d in draws], dtype=float)
    def se(a): return float(np.std(a, ddof=1)) if len(a) > 1 else np.nan

    return {
        "explained_menref":   base["explained_menref"],
        "explained_menref_se": se(col("explained_menref")),
        "explained_womenref":   base["explained_womenref"],
        "explained_womenref_se": se(col("explained_womenref")),
        "unexplained_menref":   base["unexplained_menref"],
        "unexplained_menref_se": se(col("unexplained_menref")),
        "unexplained_womenref":   base["unexplained_womenref"],
        "unexplained_womenref_se": se(col("unexplained_womenref")),
        "gap_men_minus_women":   base["gap_men_minus_women"],
    }


def pval(est, se):
    if pd.isna(est) or pd.isna(se) or se <= 0:
        return np.nan
    return 2 * (1 - norm.cdf(abs(est / se)))


In [ ]:
# ============================================================
# PHASE 1 — TabPFN once per subset (no bootstrap). Count sign flips.
# ============================================================
phase1_rows = []
for j, (label, sub) in enumerate(subsets, start=1):
    try:
        out = nonlinear_oaxaca_tabpfn(sub, Y_COL, X_COLS, GROUP_COL)
        exp_flip   = np.sign(out["explained_menref"])   != np.sign(out["explained_womenref"])
        unexp_flip = np.sign(out["unexplained_menref"]) != np.sign(out["unexplained_womenref"])
        phase1_rows.append({
            "subset": label, "n": len(sub),
            "explained_menref": out["explained_menref"],
            "explained_womenref": out["explained_womenref"],
            "unexplained_menref": out["unexplained_menref"],
            "unexplained_womenref": out["unexplained_womenref"],
            "explained_flip": bool(exp_flip),
            "unexplained_flip": bool(unexp_flip),
            "any_flip": bool(exp_flip or unexp_flip),
        })
        tag = ("E " if exp_flip else "  ") + ("U" if unexp_flip else " ")
        print(f"[{j:3d}/{len(subsets)}] {tag}  {label:30s}  n={len(sub):5d}")
    except Exception as e:
        print(f"[{j:3d}/{len(subsets)}] FAILED {label}: {e}")

phase1_df = pd.DataFrame(phase1_rows)
phase1_df.to_csv("tabpfn_phase1_signflips.csv", index=False)
n_flip_subsets = int(phase1_df["any_flip"].sum())
print(f"\nPhase 1 summary: {n_flip_subsets} / {len(phase1_df)} subsets have a sign flip.")

flip_subsets = phase1_df[phase1_df["any_flip"]]["subset"].tolist()
print(f"Flip subsets -> to bootstrap: {len(flip_subsets)}")


In [ ]:
# ============================================================
# PHASE 2 — Bootstrap on sign-flip subsets
# ============================================================
subset_dict = {lab: sub for lab, sub in subsets}
boot_rows = []

for j, label in enumerate(flip_subsets, start=1):
    sub = subset_dict[label]
    print(f"[{j}/{len(flip_subsets)}] bootstrap {label} | n={len(sub)}")
    try:
        out = bootstrap_tabpfn(sub, Y_COL, X_COLS, GROUP_COL,
                               B=CONFIG["B_boot"], random_state=1000+j,
                               n_jobs=N_JOBS_BOOT)
        row = {"subset": label, "n": len(sub),
               **{k: out[k] for k in out}}
        # p-values
        for comp in ["explained_menref", "explained_womenref",
                     "unexplained_menref", "unexplained_womenref"]:
            row[f"{comp}_pvalue"] = pval(out[comp], out[f"{comp}_se"])
        boot_rows.append(row)
        pd.DataFrame(boot_rows).to_csv("tabpfn_phase2_bootstrap.csv", index=False)
    except Exception as e:
        print(f"   FAILED: {e}")

boot_df = pd.DataFrame(boot_rows)
print(f"\nPhase 2 complete: {len(boot_df)} subsets bootstrapped.")


In [ ]:
# ============================================================
# PHASE 3 — Summary row for TabPFN (Total flips | 10% | 5% | 1%)
# Same format as icu_139_final_models.ipynb; ready to append.
# ============================================================

# Build per-component flip flags using Phase 1 point estimates (flip definition is
# based on signs only). Then use Phase 2 bootstrap p-values to flag significance.
#   - Total flips: # of flipped components across all 139 subsets (explained +
#     unexplained counted separately).
#   - α level counts: of those flipped components, how many have at least one
#     reference with p < α.

boot_lookup = boot_df.set_index("subset") if len(boot_df) else pd.DataFrame()

def sig_at(component, alpha, subset_label):
    if subset_label not in boot_lookup.index:
        return False
    r = boot_lookup.loc[subset_label]
    pm = r.get(f"{component}_menref_pvalue",   np.nan)
    pw = r.get(f"{component}_womenref_pvalue", np.nan)
    return (pd.notna(pm) and pm < alpha) or (pd.notna(pw) and pw < alpha)

total_flips = 0
counts = {a: 0 for a in CONFIG["alpha_levels"]}
for _, r in phase1_df.iterrows():
    for comp, is_flip in [("explained", r["explained_flip"]),
                          ("unexplained", r["unexplained_flip"])]:
        if not is_flip:
            continue
        total_flips += 1
        for a in CONFIG["alpha_levels"]:
            if sig_at(comp, a, r["subset"]):
                counts[a] += 1

summary_row = {
    "Model": "TabPFN",
    "Total flips": total_flips,
    "10%": counts[0.10],
    "5%":  counts[0.05],
    "1%":  counts[0.01],
}
summary_df = pd.DataFrame([summary_row])
display(summary_df)
summary_df.to_csv("tabpfn_flip_summary_row.csv", index=False)

print("\nAppend this row to the summary table produced by icu_139_final_models.ipynb.")


In [ ]:
# ============================================================
# PHASE 4 — Bootstrap TabPFN on HR quartile 2
# ============================================================
TARGET = "HR quartile 2"
sub = dict(subsets)[TARGET]
print(f"{TARGET}: n={len(sub)}  |  B={CONFIG['B_boot']}  |  n_jobs={N_JOBS_BOOT}")

out = bootstrap_tabpfn(sub, Y_COL, X_COLS, GROUP_COL,
                       B=CONFIG["B_boot"], random_state=9999,
                       n_jobs=N_JOBS_BOOT)

row = {"subset": TARGET, "n": len(sub),
       "gap_men_minus_women": out["gap_men_minus_women"]}
for comp in ["explained_menref", "explained_womenref",
             "unexplained_menref", "unexplained_womenref"]:
    row[comp]             = out[comp]
    row[f"{comp}_se"]     = out[f"{comp}_se"]
    row[f"{comp}_pvalue"] = pval(out[comp], out[f"{comp}_se"])

hrq2_row = pd.DataFrame([row])
display(hrq2_row)
hrq2_row.to_csv("tabpfn_hrq2_bootstrap.csv", index=False)

def _fmt(e, s, p):
    if pd.isna(e) or pd.isna(s): return ""
    stars = "***" if (pd.notna(p) and p < 0.01) else \
            ("**" if (pd.notna(p) and p < 0.05) else \
            ("*"  if (pd.notna(p) and p < 0.10) else ""))
    return f"{e:.3f}{stars} ({s:.3f})"

print("\nFormatted (TabPFN, HR quartile 2):")
for comp, label in [("explained_womenref",   "Explained (Women ref)"),
                    ("unexplained_womenref", "Unexplained (Women ref)"),
                    ("explained_menref",     "Explained (Men ref)"),
                    ("unexplained_menref",   "Unexplained (Men ref)")]:
    print(f"  {label:24s} = {_fmt(row[comp], row[f'{comp}_se'], row[f'{comp}_pvalue'])}")
